In [26]:
import sys
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import Subset, ConcatDataset, random_split
from torchvision import datasets, transforms
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# --- Library Imports ---
sys.path.append('../') 
try:
    from NeuroHDC.FHRR import *
    from NeuroHDC.fn import *
    from conformalHDC.models import *
    from conformalHDC.methods import *
    from conformalHDC.utils import *
except ImportError:
    try:
        from utils import eval_accuracy, eval_lc_accuracy
    except ImportError:
        print("Warning: Utils not found. Ensure '../' is in path.")

In [29]:
################======== Experiment Logic ========################
def run_single_experiment(random_state, rat_id, alpha, in_path_id):
    # Data Loading (Exact matching with exp_rat.py)
    splits = prep_loader_slicing(
        irat=rat_id, 
        split_ratio=(0.5, 0.4, 0.1), 
        in_path=in_path_id,
        step_bins=STEP_BINS,
        slicing_window=SLICING_WINDOW,
        bin_size=BIN_SIZE,
        seed=random_state
    )

    # Flatten time and feature dimensions for KNN: (n, nT, p) -> (n, nT*p)
    X_train = splits.train.X.reshape(splits.train.X.shape[0], -1)
    y_train = splits.train.y
    X_cal = splits.cal.X.reshape(splits.cal.X.shape[0], -1)
    y_cal = splits.cal.y
    X_test = splits.test.X.reshape(splits.test.X.shape[0], -1)
    y_test = splits.test.y

    unique_labels = sorted(np.unique(y_train))

    # Train KNN
    cknn = ConformalKNN(n_neighbors=5, random_state=random_state)
    cknn.fit(X_train, y_train)
    
    # Compute Calibration Scores
    cknn.compute_calib_scores(X_cal, y_cal)
    
    exp_results = []

    # Baseline Point Prediction Accuracy
    preds = cknn.model.predict(X_test)
    point_acc = eval_accuracy(preds, y_test)
    
    # Set-Valued Prediction (Marginal CP)
    sets = cknn.set_valued_CP(X_test, alpha, marginal=True)
    sizes = [len(p) for p in sets]
    covered = [1 if y in p else 0 for y, p in zip(y_test, sets)]
    
    lc_covs = []
    for lbl in unique_labels:
        lbl_idx = np.where(y_test == lbl)[0]
        if len(lbl_idx) > 0:
            lc_covs.append(np.mean([covered[i] for i in lbl_idx]))
    
    exp_results.append({
        "exp": "set_valued",
        "random_state": random_state,
        "rat_id": rat_id,
        "score_type": "inverse_quantile",
        "alpha": alpha,
        "marginal": True,
        "set_cov": np.mean(covered), 
        "set_size": np.mean(sizes), 
        "lc_covs": lc_covs if lc_covs else 0.0,
        "point_acc": point_acc
    })

    return pd.DataFrame(exp_results)

In [31]:
# Fixed Constants
EXP_NAME = "knn_odor_decoding"
REPETITIONS = 500
TRAINING_WINDOW = (200, 600)
RUNNING_WINDOW = (0, 200)
BIN_SIZE = 25
SLICING_WINDOW = 200
STEP_BINS = 2
SEED = 1
ALPHA = 0.2

path_id = Path("../data/rat") / f"odor_prep_{TRAINING_WINDOW}_{BIN_SIZE}.pickle"
out_dir = Path(f"../experiments_real/results/{EXP_NAME}")
out_dir.mkdir(parents=True, exist_ok=True)

for rat in range(5):
    outfile = out_dir / f"rat{rat}_seed{SEED}_alpha{ALPHA}.csv"
    results_list = []
    
    for i in tqdm(range(1, REPETITIONS + 1), desc=f"Repetitions Rat {rat}"):
        current_state = REPETITIONS * (SEED - 1) + i
        try:
            df_rep = run_single_experiment(current_state, rat, ALPHA, path_id)
            results_list.append(df_rep)
        except Exception as e:
            print(f"Error in state {current_state}: {e}")

    if results_list:
        final_df = pd.concat(results_list, ignore_index=True)
        final_df.to_csv(outfile, index=False)

Repetitions Rat 0:   0%|▎                                                              | 2/500 [00:00<00:39, 12.65it/s]

Original Shape: (133, 16, 92)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (665, 8, 92)
Original Shape: (133, 16, 92)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (665, 8, 92)
Original Shape: (133, 16, 92)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (665, 8, 92)


Repetitions Rat 0:   1%|▌                                                              | 4/500 [00:00<00:38, 12.76it/s]

Original Shape: (133, 16, 92)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (665, 8, 92)
Original Shape: (133, 16, 92)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (665, 8, 92)
Original Shape: (133, 16, 92)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (665, 8, 92)


Repetitions Rat 0:   2%|█                                                              | 8/500 [00:00<00:39, 12.39it/s]

...

Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)
Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)
Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)


Repetitions Rat 4:  99%|████████████████████████████████████████████████████████████▌| 496/500 [00:35<00:00, 13.13it/s]

Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)
Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)
Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)


Repetitions Rat 4: 100%|█████████████████████████████████████████████████████████████| 500/500 [00:36<00:00, 13.84it/s]

Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)
Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)


In [28]:
final_df

,exp,random_state,rat_id,score_type,alpha,marginal,set_cov,set_size,lc_covs,point_acc
0,set_valued,1,1,inverse_quantile,0.2,True,0.844444,1.755556,"[0.9833333333333333, 0.6, 0.5, 0.6]",0.711111
1,set_valued,2,1,inverse_quantile,0.2,True,0.722222,1.844444,"[1.0, 0.48, 0.4, 0.7]",0.500000
2,set_valued,3,1,inverse_quantile,0.2,True,0.733333,2.211111,"[0.9, 0.6, 0.6666666666666666, 0.7666666666666...",0.444444
3,set_valued,4,1,inverse_quantile,0.2,True,0.788889,2.033333,"[1.0, 0.76, 0.8, 0.6571428571428571]",0.477778
4,set_valued,5,1,inverse_quantile,0.2,True,0.711111,1.700000,"[0.9555555555555556, 0.26666666666666666, 0.4,...",0.533333
...,...,...,...,...,...,...,...,...,...,...
95,set_valued,96,1,inverse_quantile,0.2,True,0.833333,1.944444,"[0.92, 0.8, 0.6, 0.8571428571428571]",0.533333
96,set_valued,97,1,inverse_quantile,0.2,True,0.755556,1.811111,"[0.9333333333333333, 0.4, 0.4, 0.72]",0.588889
97,set_valued,98,1,inverse_quantile,0.2,True,0.855556,1.922222,"[0.9428571428571428, 0.84, 0.5, 0.9]",0.577778
98,set_valued,99,1,inverse_quantile,0.2,True,0.711111,1.866667,"[0.9333333333333333, 0.55, 0.7, 0.6]",0.533333
